In [2]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic
client = Anthropic()

model = "claude-sonnet-4-0"


In [4]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
messages = []

add_user_message(messages, "Write a one sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011XojxAmF3a8o5mTf4U1n1T', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeUserDB is a mock database containing 10,000 randomly generated user profiles with fabricated names', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=', email 

In [6]:
# better streaming approach useing the SDK
messages = []

add_user_message(messages, "Write a one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()

ParsedMessage(id='msg_019gUEnQuzNkU9Z7kGDweQAU', container=None, content=[ParsedTextBlock(citations=None, text='A fake database is a simulated or mock data storage system that contains fabricated, placeholder, or test data rather than real production information, commonly used for development, testing, or demonstration purposes.', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=42, server_tool_use=None, service_tier='standard'))

In [7]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
print(text)


{
  "Name": "OrderProcessingRule",
  "EventPattern": {
    "source": ["ecommerce.orders"],
    "detail-type": ["Order Placed"]
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"
    }
  ]
}



In [8]:
# Exercise
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")
text = chat(messages, stop_sequences=["```"])

text.strip()

'aws s3 ls\naws lambda list-functions\naws iam list-users'